# genulens Python API examples

This notebook demonstrates the direct Python API for `genulens`. The API calls the shared C++ simulation core and returns typed result objects; it does not run `./genulens` or parse command-line stdout.

The examples cover plain sampling, typed configuration, CLI-like output labels, forward source annotations, and custom likelihoods.


In [ ]:
from pathlib import Path
import sys
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Find build/ whether Jupyter is launched from the repository root or examples/.
repo_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
build_dir = repo_root / "build"
if build_dir.exists():
    sys.path.insert(0, str(build_dir))

import genulens


## Helper functions

`SimulationResult` carries column names and can be converted to a NumPy array. For analysis, a pandas DataFrame is usually the most convenient representation.


In [ ]:
def as_dataframe(result):
    return pd.DataFrame(result.to_numpy(), columns=result.columns)


def weighted_quantile(values, weights, q):
    values = np.asarray(values)
    weights = np.asarray(weights)
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cdf = np.cumsum(weights)
    cdf /= cdf[-1]
    return np.interp(q, cdf, values)


def summarize_events(df, label):
    w = df["wtj"].to_numpy()
    return pd.Series(
        {
            "n_events": len(df),
            "weight_sum": w.sum(),
            "tE_p16": weighted_quantile(df["t_E"], w, 0.16),
            "tE_p50": weighted_quantile(df["t_E"], w, 0.50),
            "tE_p84": weighted_quantile(df["t_E"], w, 0.84),
            "mass_p50": weighted_quantile(df["M_L"], w, 0.50),
            "thetaE_p50": weighted_quantile(df["theta_E"], w, 0.50),
        },
        name=label,
    )


## 1. Plain sampling

Create `Config(l, b, n_simu, seed)` and call `genulens.simulate()`. `n_simu` is the target number of accepted events returned by the simulation.

The default Python columns use the `VERBOSITY=3` style labels and ordering. They include `mu_rel_N` and `mu_rel_E` directly, so callers do not need to reconstruct them from parallax components.


In [ ]:
cfg = genulens.Config(l=1.0, b=-3.9, n_simu=20_000, seed=42)

result = genulens.simulate(cfg)
df = as_dataframe(result)

result.columns


In [ ]:
display_columns = [
    "wtj",
    "t_E",
    "theta_E",
    "pi_E",
    "pi_EN",
    "pi_EE",
    "mu_rel",
    "mu_rel_N",
    "mu_rel_E",
    "M_L",
]
df[display_columns].head()


In [ ]:
summarize_events(df, "plain sampling")

bins_tE = np.logspace(-1, 3, 60)
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df["t_E"], bins=bins_tE, weights=df["wtj"], density=True, histtype="stepfilled", alpha=0.35)
ax.axvline(weighted_quantile(df["t_E"], df["wtj"], 0.50), color="black", lw=1)
ax.set_xscale("log")
ax.set_xlabel("tE [day]")
ax.set_ylabel("weighted density")
ax.set_title("Plain sampling")
plt.show()


## 2. Corner plot of physical event parameters

A compact corner plot is useful for checking the joint prior over lens mass, lens/source distances, and the relative proper-motion components.


In [ ]:
corner_columns = ["M_L", "D_L", "D_S", "mu_rel_N", "mu_rel_E"]

mu_abs = np.sqrt(df["mu_rel_N"]**2 + df["mu_rel_E"]**2)

df_corner = df[
    (mu_abs <= 20) &
    (df["M_L"] <= 1.5)
].copy()

corner_values = df_corner[corner_columns].to_numpy()
corner_weights = df_corner["wtj"].to_numpy()

try:
    import corner

    fig = corner.corner(
        corner_values,
        labels=corner_columns,
        weights=corner_weights,
        quantiles=[0.16, 0.50, 0.84],
        show_titles=True,
        title_fmt=".3g",
    )
except ImportError:
    axes = pd.plotting.scatter_matrix(
        df_corner[corner_columns],
        figsize=(9, 9),
        diagonal="hist",
        hist_kwds={"weights": corner_weights, "bins": 40},
        alpha=0.08,
    )
    fig = axes[0, 0].figure

fig.suptitle("Plain sampling joint prior, |mu_rel| <= 20 & M_L <= 1.5", y=1.02)
plt.show()

## 3. Selecting a different verbosity

The default output already uses the `VERBOSITY=3` style layout. Set `cfg.sampling.verbosity` when you need another CLI-style table shape.


In [ ]:
cfg_v1 = genulens.Config(l=1.0, b=-3.9, n_simu=5_000, seed=42)
cfg_v1.sampling.verbosity = 1

result_v1 = genulens.simulate(cfg_v1)
df_v1 = as_dataframe(result_v1)

result_v1.columns


In [ ]:
df_v1[[
    "wtj",
    "tE",
    "thetaE",
    "piE",
    "M_L",
    "D_S",
    "D_L",
    "mu_rel",
]].head()


## 4. Built-in observation constraints through typed config

Observation constraints such as `tE` and `thetaE` are set directly on `cfg.observation`. These values are passed as typed C++ configuration fields, not as command-line option strings.


In [ ]:
cfg_obs = genulens.Config(l=1.0, b=-3.9, n_simu=20_000, seed=42)
cfg_obs.observation.tE_obs = 54.5
cfg_obs.observation.tE_err = 5.0
cfg_obs.observation.thetaE_obs = 0.55
cfg_obs.observation.thetaE_err = 0.15

result_obs = genulens.simulate(cfg_obs)
df_obs = as_dataframe(result_obs)

pd.concat([
    summarize_events(df, "plain"),
    summarize_events(df_obs, "with observation constraints"),
], axis=1)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(df["t_E"], bins=bins_tE, weights=df["wtj"], density=True, histtype="step", label="plain")
axes[0].hist(df_obs["t_E"], bins=bins_tE, weights=df_obs["wtj"], density=True, histtype="step", label="constrained")
axes[0].set_xscale("log")
axes[0].set_xlabel("tE [day]")
axes[0].legend()

bins_theta = np.logspace(-3, 1, 60)
axes[1].hist(df["theta_E"], bins=bins_theta, weights=df["wtj"], density=True, histtype="step", label="plain")
axes[1].hist(df_obs["theta_E"], bins=bins_theta, weights=df_obs["wtj"], density=True, histtype="step", label="constrained")
axes[1].set_xscale("log")
axes[1].set_xlabel("thetaE [mas]")
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. Forward source annotations

`cfg.forward_source.enabled = 1` appends source stellar properties from the shared isochrone lookup. This is an annotation mode for forward-prior studies; it does not replace the legacy LF/CMF source-selection and event-rate weighting.


In [ ]:
cfg_src = genulens.Config(l=1.0, b=-3.9, n_simu=5_000, seed=42)
cfg_src.forward_source.enabled = 1
cfg_src.forward_source.photometry = "roman"
cfg_src.forward_source.min_initial_mass_msun = 0.1
cfg_src.forward_source.max_initial_mass_msun = 1.0
cfg_src.forward_source.match_source_selection = 1
cfg_src.forward_source.selection_bands = ["F146mag"]
cfg_src.forward_source.selection_min_magnitudes = [17.0]
cfg_src.forward_source.selection_max_magnitudes = [24.0]

result_src = genulens.simulate(cfg_src)
df_src = as_dataframe(result_src)

[c for c in result_src.columns if c.startswith("source_")]


In [ ]:
source_columns = [
    "M_S_ini",
    "M_S",
    "R_S",
    "teff_S",
    "logg_S",
    "theta_S",
    "M_F146mag_S",
]
df_src[source_columns].head()


## 6. Model, source-selection, and sampling config

Common model parameters are exposed under `cfg.model.*`. Source-selection and extinction settings live under `cfg.source.*`, while Monte Carlo behavior is controlled by `cfg.sampling.*`.


In [ ]:
cfg_model = genulens.Config(l=0.5, b=-2.0, n_simu=10_000, seed=7)

# Source selection and extinction
cfg_model.source.i_min = 14.0
cfg_model.source.i_max = 21.0
cfg_model.source.ai_rc = 1.5
cfg_model.source.evi_rc = 1.2

# Model parameters
cfg_model.model.imf.alpha2 = -1.35
cfg_model.model.density.stellar_halo = 0
cfg_model.model.nsd.enabled = 0
cfg_model.model.kinematics.omega_p = 45.0

# Sampling options
cfg_model.sampling.binary = 1
cfg_model.sampling.remnant = 1
cfg_model.sampling.small_gamma = 1
cfg_model.sampling.n_like_min = 2_000

result_model = genulens.simulate(cfg_model)
df_model = as_dataframe(result_model)
summarize_events(df_model, "configured model")


## 7. Custom likelihood with multiple terms

Pass a Python callable as `likelihood=` to evaluate additional likelihood terms inside the `EventSampler` loop. The callable receives an `Event` and returns a multiplicative weight.

A return value less than or equal to zero rejects the event. Keep hard cuts broad enough that the sampler can still accept events efficiently.


In [ ]:
def gaussian(x, mu, sigma):
    z = (x - mu) / sigma
    return math.exp(-0.5 * z * z)


def complex_likelihood(event):
    if event.t_E < 5.0 or event.t_E > 300.0:
        return 0.0
    if event.D_L >= event.D_S:
        return 0.0

    like = gaussian(math.log10(event.t_E), math.log10(54.5), 0.18)
    like *= gaussian(event.theta_E, 0.55, 0.18)
    like *= gaussian(math.log10(event.M_L), math.log10(0.35), 0.45)
    return like


cfg_custom = genulens.Config(l=1.0, b=-3.9, n_simu=10_000, seed=11)
result_custom = genulens.simulate(cfg_custom, likelihood=complex_likelihood)
df_custom = as_dataframe(result_custom)

pd.concat([
    summarize_events(df, "plain"),
    summarize_events(df_custom, "custom likelihood"),
], axis=1)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df["t_E"], bins=bins_tE, weights=df["wtj"], density=True, histtype="step", label="plain")
axes[0].hist(df_custom["t_E"], bins=bins_tE, weights=df_custom["wtj"], density=True, histtype="step", label="custom")
axes[0].set_xscale("log")
axes[0].set_xlabel("tE [day]")

axes[1].hist(df["theta_E"], bins=bins_theta, weights=df["wtj"], density=True, histtype="step", label="plain")
axes[1].hist(df_custom["theta_E"], bins=bins_theta, weights=df_custom["wtj"], density=True, histtype="step", label="custom")
axes[1].set_xscale("log")
axes[1].set_xlabel("thetaE [mas]")

bins_mass = np.logspace(-2, 1, 60)
axes[2].hist(df["M_L"], bins=bins_mass, weights=df["wtj"], density=True, histtype="step", label="plain")
axes[2].hist(df_custom["M_L"], bins=bins_mass, weights=df_custom["wtj"], density=True, histtype="step", label="custom")
axes[2].set_xscale("log")
axes[2].set_xlabel("lens mass [Msun]")

for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()


## 8. Weighted contribution by lens component

`iL` and `iS` are included in the default result columns as lens and source component IDs. Component-level weighted fractions are useful for checking which Galactic population is selected by a likelihood model.


In [ ]:
component_names = {
    0: "thin disk 0", 1: "thin disk 1", 2: "thin disk 2", 3: "thin disk 3",
    4: "thin disk 4", 5: "thin disk 5", 6: "thin disk 6",
    7: "thick disk", 8: "bar", 9: "NSD", 10: "stellar halo",
}


def component_fractions(df, column):
    grouped = df.groupby(column)["wtj"].sum()
    fractions = grouped / grouped.sum()
    out = fractions.rename(index=lambda x: component_names.get(int(x), f"component {int(x)}"))
    return out.sort_values(ascending=False)

component_fractions(df_custom, "iL").head(8)


## 9. `ruc` shortcut

For quick experiments, `genulens.ruc(l, b, n_simu, seed)` runs a simulation without explicitly constructing a `Config`. Use `Config` for workflows with observation constraints, source annotations, model parameters, or sampling options.


In [ ]:
quick = genulens.ruc(l=0.0, b=-2.0, n_simu=2_000, seed=1)
quick_df = as_dataframe(quick)
quick_df[[
    "wtj",
    "t_E",
    "theta_E",
    "M_L",
    "mu_rel",
    "mu_rel_N",
    "mu_rel_E",
]].describe()
